In [1]:
import os

print("INPUT ROOT:")
print(os.listdir("/kaggle/input"))

print("\nDATASETS:")
print(os.listdir("/kaggle/input/datasets"))

print("\nSAUTKIN DATASETS:")
print(os.listdir("/kaggle/input/datasets/sautkin"))

INPUT ROOT:
['datasets']

DATASETS:
['sautkin']

SAUTKIN DATASETS:
['imagenet1k1', 'imagenet1kvalid', 'imagenet1k2', 'imagenet1k0', 'imagenet1k3']


## Clean the Working Directory

In [2]:
import shutil
import os

for item in os.listdir("/kaggle/working"):
    path = os.path.join("/kaggle/working", item)

    if os.path.isdir(path):
        shutil.rmtree(path)
    else:
        os.remove(path)

In [3]:
!rm -rf LSNET-advanced
!git clone -b proposal_8 https://github.com/Param45/LSNET-advanced.git
%cd LSNET-advanced
!ls

Cloning into 'LSNET-advanced'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 304 (delta 8), reused 8 (delta 4), pack-reused 285 (from 2)
Receiving objects: 100% (304/304), 41.57 MiB | 42.36 MiB/s, done.
Resolving deltas: 100% (127/127), done.
/kaggle/working/LSNET-advanced
data		  logs			       README_robustness.md
detection	  losses.py		       requirements.txt
engine.py	  lsnet_modification_guide.md  robust.py
eval_robust.sh	  lsnet-p3.ipynb	       robust_utils.py
eval.sh		  lsnet-p7.ipynb	       segmentation
figures		  lsnet-p8.ipynb	       speed.py
flops.py	  main.py		       train.sh
kaggle_config.py  model			       utils.py
kaggle_run.py	  pretrain
KAGGLE_SETUP.md   README.md


In [4]:
import torch
print(torch.__version__)

2.10.0+cu128


In [5]:
!pip install -q timm fvcore wandb
!pip install timm==0.5.4 einops==0.4.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 23.2 MB/s eta 0:00:00
  Attempting uninstall: einops
    Found existing installation: einops 0.8.2
    Uninstalling einops-0.8.2:
      Successfully uninstalled einops-0.8.2
  Attempting uninstall: timm
    Found existing installation: timm 1.0.26
    Uninstalling timm-1.0.26:
      Successfully uninstalled timm-1.0.26


In [6]:
from kaggle_config import IMAGENET_PATH

print(IMAGENET_PATH)

/kaggle/input/datasets/sautkin


In [7]:
import os

for ds in [
    "imagenet1k0",
    "imagenet1k1",
    "imagenet1k2",
    "imagenet1k3",
    "imagenet1kvalid"
]:
    path = f"/kaggle/input/datasets/sautkin/{ds}"
    print(ds, len(os.listdir(path)))

imagenet1k0 500
imagenet1k1 500
imagenet1k2 500
imagenet1k3 500
imagenet1kvalid 1000


In [8]:
!python main.py --help

usage: main.py [-h] [--batch-size BATCH_SIZE] [--epochs EPOCHS]
               [--model MODEL] [--input-size INPUT_SIZE] [--model-ema]
               [--no-model-ema] [--model-ema-decay MODEL_EMA_DECAY]
               [--model-ema-steps MODEL_EMA_STEPS] [--model-ema-force-cpu]
               [--opt OPTIMIZER] [--opt-eps EPSILON]
               [--opt-betas BETA [BETA ...]] [--clip-grad NORM]
               [--clip-mode CLIP_MODE] [--momentum M]
               [--weight-decay WEIGHT_DECAY] [--sched SCHEDULER] [--lr LR]
               [--lr-noise pct, pct [pct, pct ...]] [--lr-noise-pct PERCENT]
               [--lr-noise-std STDDEV] [--warmup-lr LR] [--min-lr LR]
               [--decay-epochs N] [--warmup-epochs N] [--cooldown-epochs N]
               [--patience-epochs N] [--decay-rate RATE] [--ThreeAugment]
               [--sparse-ska] [--sparse-top-k SPARSE_TOP_K] [--group-se]
               [--use-shifted-ska] [--dataset-fraction DATASET_FRACTION]
               [--color-jitter PC

## Verify Dataset Loading (40% fraction)

Proposal 8 uses only 40% of the ImageNet-1k training data.
The `--dataset-fraction 0.4` flag is passed to `main.py` which
passes it to `build_dataset`, which randomly subsamples 40% of
the training files (fixed seed=42) before building the dataset object.
Validation set is always loaded in full.

In [9]:
from data.datasets import build_dataset

In [10]:
from types import SimpleNamespace
import time

args = SimpleNamespace(
    data_set="IMNET",
    data_path="/kaggle/input/datasets/sautkin",
    input_size=224,
    color_jitter=0.4,
    aa='rand-m9-mstd0.5-inc1',
    train_interpolation='bicubic',
    reprob=0.25,
    remode='pixel',
    recount=1,
    finetune='',
    inat_category='name',
    dataset_fraction=0.4
)

start = time.time()

dataset_train, n_classes = build_dataset(
    is_train=True,
    args=args
)

print("Classes:", n_classes)
print("Samples:", len(dataset_train))
print("Time:", time.time() - start)

Classes: 1000
Samples: 512118
Time: 33.571417570114136


In [11]:
start = time.time()

dataset_val, n_classes = build_dataset(
    is_train=False,
    args=args
)

print("Classes:", n_classes)
print("Samples:", len(dataset_val))
print("Time:", time.time() - start)

Classes: 1000
Samples: 50000
Time: 19.698701858520508


In [12]:
!python kaggle_run.py --help

usage: kaggle_run.py [-h] --action
                     {train_clf,eval_clf,train_det,test_det,train_seg,test_seg,robust_clf}
                     [--model MODEL] [--config CONFIG]
                     [--checkpoint CHECKPOINT] [--resume RESUME]
                     [--extra-args EXTRA_ARGS]

Kaggle execution entrypoint helper

options:
  -h, --help            show this help message and exit
  --action {train_clf,eval_clf,train_det,test_det,train_seg,test_seg,robust_clf}
                        Action to perform
  --model MODEL         Model name
  --config CONFIG       Path to MMCV/MMDet/MMSeg config file
  --checkpoint CHECKPOINT
                        Path to checkpoint file
  --resume RESUME       Path to checkpoint to resume training from
  --extra-args EXTRA_ARGS
                        Extra arguments to pass to the script


In [13]:
import os
os.environ["WANDB_MODE"] = "disabled"

## Run Fine-Tuning — Proposal 8 (Shifted-Window SKA, 40% Dataset)

**What this does:**
- Loads the pretrained `lsnet_t.pth` weights with `strict=True` — ShiftedSKA adds zero new parameters so all keys match exactly.
- Every `LSConv` block at `depth % 4 == 3` uses a shifted-window SKA (cyclic roll by 1 pixel before the 3×3 aggregation, then unroll after).
- Adjacent blocks (regular + shifted) together tile the spatial neighbourhood.
- BN running stats for shifted blocks adapt within 2–5 epochs.
- Trains for 14 epochs with 2 warmup epochs on 40% of ImageNet-1k.

In [14]:
# Fine-tune for 14 epochs for Proposal 8 with 40% dataset

%cd /kaggle/working/LSNET-advanced
!python kaggle_run.py --action train_clf --model lsnet_t --finetune /kaggle/working/LSNET-advanced/pretrain/lsnet_t.pth --epochs 14 --warmup-epochs 2 --lr 5e-5 --use-shifted-ska --dataset-fraction 0.4


/kaggle/working/LSNET-advanced
Executing: torchrun --nproc_per_node=2 main.py --model lsnet_t --finetune /kaggle/working/LSNET-advanced/pretrain/lsnet_t.pth --epochs 14 --warmup-epochs 2 --lr 5e-5 --use-shifted-ska --dataset-fraction 0.4
W0722 07:03:30.337000 141 torch/distributed/run.py:852]
W0722 07:03:30.337000 141 torch/distributed/run.py:852] *****************************************
W0722 07:03:30.337000 141 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
W0722 07:03:30.337000 141 torch/distributed/run.py:852] *****************************************
| distributed init (rank 1): env://
| distributed init (rank 0): env://
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `